# 10장. 분류 분석으로 주문 취소 여부 예측하기

이번 과제는 높은 Accuracy 자체보다 **타깃 정의 → 예측 시점 → 누수 방지 → Validation에서 모델/Threshold 선택 → Final Test 독립성 → FP/FN 해석 → Privacy-safe Output**의 순서를 지키는 것이 핵심입니다.

- `completed = 0`
- `cancelled = 1`
- `refunded` 및 기타 상태는 모델링 대상에서 제외
- 모델과 Threshold는 Validation에서 선택
- Final Test는 모든 선택을 고정한 뒤 마지막 평가에만 사용

### 제출 정보
- GitHub ID: `cyw0927`
- 작성일: 2026-10-01
- 최종 Notebook: `assignments/chapter10/chapter10.ipynb`

## 0. 실행 전 확인

Chapter 10은 Chapter 05 전용 오류 데이터가 아니라 공통 프로젝트 `data/raw` → `data/processed` 흐름을 사용합니다.

필요하면 프로젝트 루트에서 다음을 실행합니다.

```powershell
python scripts/prepare_ch10_data.py
```

이 Notebook은 `customers_clean.csv`, `orders_clean.csv`, `order_items_clean.csv`를 사용합니다.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("프로젝트 루트를 찾지 못했습니다.")

PROJECT_ROOT = find_project_root(Path.cwd())
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"
ASSIGNMENT_DIR = PROJECT_ROOT / "assignments" / "chapter10"
IMAGE_DIR = ASSIGNMENT_DIR / "images"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Python:", sys.executable)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("IMAGE_DIR:", IMAGE_DIR)

Python: /opt/hostedtoolcache/Python/3.12.14/x64/bin/python
PROJECT_ROOT: /home/runner/work/-llm-data-analysis-course/-llm-data-analysis-course
PROCESSED_DIR: /home/runner/work/-llm-data-analysis-course/-llm-data-analysis-course/data/processed
IMAGE_DIR: /home/runner/work/-llm-data-analysis-course/-llm-data-analysis-course/assignments/chapter10/images


## 1. 분류 문제 정의

- **타깃:** 주문 취소 여부 `is_cancelled`
- **0:** completed
- **1:** cancelled
- **제외:** refunded 및 기타 상태
- **예측 단위:** 주문 1건
- **예측 시점:** 주문 생성 직후라고 가정
- **사용 가능 정보:** 주문 생성 시점의 고객 비식별 특성, 결제수단, 주문 상세를 주문 단위로 집계한 수량·금액 정보, 주문 날짜에서 만든 파생 정보
- **제외 정보:** `order_status`, `is_cancelled`, 각종 ID, 취소 사유·취소 시각처럼 결과 이후에 생기는 정보

### 나의 해석과 판단

주문 취소 여부를 예측하면서 이미 확정된 주문 상태나 취소 이후 정보를 입력하면 정답을 미리 알려주는 데이터 누수가 발생합니다. 따라서 실제 운영 시점을 주문 생성 직후로 고정하고, 그 시점에 알 수 있다고 가정한 정보만 사용합니다.

In [2]:
from src.classification import (
    FORBIDDEN_FEATURES,
    TARGET_COLUMN,
    build_classification_dataset,
    build_feature_audit,
    build_split_summary,
    confusion_matrix_dataframe,
    create_prediction_result,
    final_test_evaluation,
    load_classification_source_data,
    public_prediction_result,
    run_classification_analysis,
    select_validation_model,
    split_train_validation_test,
    target_distribution,
    threshold_metrics,
    choose_threshold,
    train_and_compare_on_validation,
    build_classification_validation,
)

data = load_classification_source_data(PROCESSED_DIR)
customers = data["customers"]
orders = data["orders"]
order_items = data["order_items"]

for name, frame in data.items():
    print(name, frame.shape)
print("\n주문 상태:")
display(orders["order_status"].value_counts(dropna=False).rename_axis("order_status").reset_index(name="count"))

customers (150, 6)
orders (300, 7)
order_items (764, 6)

주문 상태:


,order_status,count
0,completed,184
1,cancelled,64
2,refunded,52


## 2. Feature Contract와 Leakage Audit

### 금지 Feature

```text
order_status
is_cancelled
order_id
customer_id
product_id
cancel_reason
cancelled_at
```

파일에 컬럼이 존재한다는 이유만으로 모델에 넣을 수 있는 것은 아닙니다. **예측 시점보다 뒤에 생기는 정보와 식별자, 타깃 자체**는 모델 입력에서 제외해야 합니다.

In [3]:
(
    model_data,
    numeric_features,
    categorical_features,
    merge_checks,
    data_quality_checks,
) = build_classification_dataset(
    customers=customers,
    orders=orders,
    order_items=order_items,
)

feature_audit = build_feature_audit(numeric_features, categorical_features)

print("숫자형 Feature:", numeric_features)
print("범주형 Feature:", categorical_features)
print("모델링 데이터:", model_data.shape)
display(feature_audit)

selected_features = numeric_features + categorical_features
leakage_overlap = sorted(set(selected_features) & FORBIDDEN_FEATURES)
assert not leakage_overlap
print("Leakage Audit: PASS")

숫자형 Feature: ['age', 'item_count', 'total_quantity', 'order_amount', 'order_month', 'order_dayofweek', 'days_since_signup']
범주형 Feature: ['gender', 'city', 'payment_method']
모델링 데이터: (248, 16)


,column,selected,role,reason
0,age,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
1,item_count,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
2,total_quantity,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
3,order_amount,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
4,order_month,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
5,order_dayofweek,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
6,days_since_signup,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
7,gender,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
8,city,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
9,payment_method,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정


Leakage Audit: PASS


## 3. 주문 단위 특징과 병합 검증

예측 단위가 주문 1건이므로 주문 상세를 주문 단위로 집계합니다.

- `item_count`
- `total_quantity`
- `order_amount`

먼저 `line_total = quantity × unit_price` 관계를 확인한 뒤 집계합니다. 병합은 주문 ↔ 주문특징은 **one_to_one**, 주문 ↔ 고객은 **many_to_one** 관계를 기대합니다.

In [4]:
items_check = order_items.copy()
expected_line_total = (
    pd.to_numeric(items_check["quantity"], errors="coerce")
    * pd.to_numeric(items_check["unit_price"], errors="coerce")
)
if "line_total" in items_check.columns:
    actual_line_total = pd.to_numeric(items_check["line_total"], errors="coerce")
    line_total_mismatch_count = int((actual_line_total - expected_line_total).abs().gt(1e-6).sum())
else:
    line_total_mismatch_count = 0

order_feature_duplicate_count = int(model_data["order_id"].duplicated().sum())

print("line_total 계산 불일치:", line_total_mismatch_count)
print("모델 데이터 order_id 중복:", order_feature_duplicate_count)
display(merge_checks)
display(data_quality_checks)

assert line_total_mismatch_count == 0
assert order_feature_duplicate_count == 0
assert merge_checks["status"].eq("PASS").all()
print("주문 단위 특징 / 병합 검증: PASS")

line_total 계산 불일치: 0
모델 데이터 order_id 중복: 0


,merge,before_rows,after_rows,row_count_preserved,unmatched_count,status
0,order_id → order_item_features,248,248,True,0,PASS
1,customer_id → customers,248,248,True,0,PASS


,check_item,count
0,binary_target_rows,248
1,completed_rows,184
2,cancelled_rows,64
3,excluded_status_rows,52
4,line_total_mismatch,0
5,negative_days_since_signup,45
6,missing_order_item_features,0
7,missing_customer_match,0
8,status_scope:completed,184
9,status_scope:cancelled,64


주문 단위 특징 / 병합 검증: PASS


### 추가 데이터 품질 확인

공통 Raw 데이터에서 이진 분류 대상 248건 중 **45건**은 `signup_date > order_date`로 시간 관계가 맞지 않았습니다.

이 경우 원본 날짜를 임의로 수정하거나 해당 주문을 삭제하지 않고, **`days_since_signup` 파생값만 결측으로 표시**했습니다. 이후 숫자형 Pipeline의 median imputation은 Train 데이터에서만 학습됩니다.

실제 운영 데이터라면 모델링 전에 원천 시스템에서 가입일과 주문일을 먼저 정정해야 합니다.

### 결과 관찰 및 판단

- `line_total = quantity × unit_price` 불일치: **0건**
- 주문 단위 `order_id` 중복: **0건**
- 주문 ↔ 주문 특징 병합: **248 → 248행, 미매칭 0건, PASS**
- 주문 ↔ 고객 병합: **248 → 248행, 미매칭 0건, PASS**
- 시간 모순으로 `days_since_signup`을 결측 처리한 행: **45건**

미매칭이 발생했는데 원인을 확인하지 않고 `fillna(0)`으로 채우면 실제로 연결이 끊어진 데이터 문제를 정상적인 0 값처럼 위장하게 됩니다. 따라서 행 수 보존과 미매칭 건수를 먼저 검증하고, 날짜 모순처럼 별도의 품질 문제는 Evidence에 남겨야 합니다.

## 4. 클래스 비율과 Dummy Baseline

모델링 대상은 총 **248건**입니다.

| 클래스 | 의미 | 개수 | 비율 |
|---:|---|---:|---:|
| 0 | completed | 184 | 74.19% |
| 1 | cancelled | 64 | 25.81% |

`refunded` 52건은 이진 분류 범위에서 제외했습니다.

완료 주문이 약 74%이므로 가장 많은 클래스만 예측하는 Dummy 모델도 Validation Accuracy **0.74**를 얻습니다. 하지만 취소 주문에 대한 Recall과 F1은 모두 **0**이므로 Accuracy만으로 모델을 평가하면 안 됩니다.

In [5]:
target_dist = target_distribution(model_data)
display(target_dist)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(target_dist["class_label"], target_dist["count"])
ax.set_title("Chapter 10 Target Class Distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Count")
fig.tight_layout()
fig.savefig(IMAGE_DIR / "step02_class_ratio.png", dpi=150)
plt.show()

,is_cancelled,class_label,count,ratio
0,0,completed,184,0.7419
1,1,cancelled,64,0.2581


## 5. Train / Validation / Test 역할

| Split | 역할 | completed | cancelled | 합계 |
|---|---|---:|---:|---:|
| Train | 파라미터 학습 | 110 | 38 | 148 |
| Validation | 후보 모델·Threshold 선택 | 37 | 13 | 50 |
| Test | 선택 완료 후 마지막 평가 | 37 | 13 | 50 |

세 Split 모두 completed 약 74%, cancelled 약 26%로 클래스 비율을 유지했습니다.

이번 실습은 교육용으로 stratified random split을 사용합니다. 실제 미래 주문을 예측하는 운영 환경에서는 **시간 순서 기반 out-of-time 평가**를 추가해야 합니다.

Test를 모델이나 Threshold 선택에 사용하면 Test에 맞춘 선택이 되어 최종 평가의 독립성이 깨집니다.

In [6]:
(
    X_train,
    X_valid,
    X_test,
    y_train,
    y_valid,
    y_test,
    features,
) = split_train_validation_test(
    model_data=model_data,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    random_state=42,
)

split_summary = build_split_summary(y_train, y_valid, y_test)
display(split_summary)
print("Train:", X_train.shape, "Validation:", X_valid.shape, "Final Test:", X_test.shape)

,split,is_cancelled,class_label,count,ratio
0,train,0,completed,110,0.7432
1,train,1,cancelled,38,0.2568
2,validation,0,completed,37,0.7400
3,validation,1,cancelled,13,0.2600
4,test,0,completed,37,0.7400
5,test,1,cancelled,13,0.2600


Train: (148, 10) Validation: (50, 10) Final Test: (50, 10)


## 6. Pipeline 전처리와 Validation 모델 비교

- 숫자형: median imputation → StandardScaler
- 범주형: most_frequent imputation → OneHotEncoder
- 전처리는 Pipeline 안에서 **Train으로만 fit**
- 후보: Dummy Most Frequent, Logistic Regression, Random Forest

Validation의 **F1 → Recall → Precision** 순으로 비베이스라인 모델을 고정합니다.

In [7]:
(
    models,
    validation_comparison,
    validation_predictions,
    validation_probabilities,
) = train_and_compare_on_validation(
    X_train=X_train,
    X_valid=X_valid,
    y_train=y_train,
    y_valid=y_valid,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    random_state=42,
)

display(validation_comparison)

selected_model_name = select_validation_model(validation_comparison)
selected_model = models[selected_model_name]
print("Validation에서 고정한 모델:", selected_model_name)

plot_df = validation_comparison.set_index("model")[["accuracy", "precision", "recall", "f1"]]
fig, ax = plt.subplots(figsize=(9, 5))
plot_df.plot(kind="bar", ax=ax)
ax.set_title("Validation Model Comparison")
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.tick_params(axis="x", rotation=15)
fig.tight_layout()
fig.savefig(IMAGE_DIR / "step03_baseline.png", dpi=150)
plt.show()

,model,evaluation_split,accuracy,precision,recall,f1
0,Logistic Regression,validation,0.66,0.388889,0.538462,0.451613
1,Random Forest,validation,0.66,0.300000,0.230769,0.260870
2,Dummy Most Frequent,validation,0.74,0.000000,0.000000,0.000000


Validation에서 고정한 모델: Logistic Regression


### 결과 관찰 및 판단

Validation 결과는 다음과 같습니다.

| 모델 | Accuracy | Precision | Recall | F1 |
|---|---:|---:|---:|---:|
| Dummy Most Frequent | 0.740 | 0.000 | 0.000 | 0.000 |
| Logistic Regression | 0.660 | 0.389 | 0.538 | 0.452 |
| Random Forest | 0.660 | 0.300 | 0.231 | 0.261 |

Dummy는 Accuracy는 가장 높지만 취소 주문을 하나도 잡지 못했습니다. 비베이스라인 후보 중에서는 **Logistic Regression**이 F1과 Recall이 더 높아 Validation에서 선택되었습니다.

즉 이번 문제에서는 단순 Accuracy보다 취소 클래스 탐지 성능을 함께 보는 것이 중요했습니다.

## 7. Validation Threshold 선택

Threshold를 낮추면 취소 예측이 늘어 **Recall이 올라갈 수 있지만 FP도 늘 수 있고**, 높이면 반대로 Precision이 올라가면서 FN이 늘 수 있습니다.

Threshold 역시 Final Test를 보지 않고 Validation에서 고정합니다.

In [8]:
validation_proba = validation_probabilities[selected_model_name]
threshold_df = threshold_metrics(y_valid, validation_proba)
selected_threshold = choose_threshold(threshold_df)

print("선택 모델:", selected_model_name)
print("선택 Threshold:", selected_threshold)
display(threshold_df.sort_values(["f1", "recall", "precision"], ascending=False).head(10))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(threshold_df["threshold"], threshold_df["precision"], marker="o", label="Precision")
ax.plot(threshold_df["threshold"], threshold_df["recall"], marker="o", label="Recall")
ax.plot(threshold_df["threshold"], threshold_df["f1"], marker="o", label="F1")
ax.axvline(selected_threshold, linestyle="--", label=f"Selected={selected_threshold:.2f}")
ax.set_title("Validation Threshold Comparison")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.legend()
fig.tight_layout()
fig.savefig(IMAGE_DIR / "step05_threshold.png", dpi=150)
plt.show()

선택 모델: Logistic Regression
선택 Threshold: 0.2


,threshold,accuracy,precision,recall,f1
0,0.20,0.38,0.295455,1.000000,0.456140
9,0.65,0.76,0.555556,0.384615,0.454545
6,0.50,0.66,0.388889,0.538462,0.451613
1,0.25,0.40,0.292683,0.923077,0.444444
4,0.40,0.58,0.333333,0.615385,0.432432
7,0.55,0.68,0.400000,0.461538,0.428571
3,0.35,0.50,0.300000,0.692308,0.418605
2,0.30,0.44,0.285714,0.769231,0.416667
5,0.45,0.60,0.333333,0.538462,0.411765
8,0.60,0.70,0.416667,0.384615,0.400000


### 선택 결과와 이유

Validation에서 자동 선택된 Threshold는 **0.20**입니다.

대표 비교:
- 0.20 → Precision **0.295**, Recall **1.000**, F1 **0.456**
- 0.50 → Precision **0.389**, Recall **0.538**, F1 **0.452**
- 0.65 → Precision **0.556**, Recall **0.385**, F1 **0.455**

자동 규칙은 Validation의 **F1 → Recall → Precision → 더 낮은 Threshold** 순입니다. 0.20이 F1에서 근소하게 가장 높았고 Recall은 1.0이어서 선택되었습니다.

다만 Threshold가 낮아질수록 FP가 크게 늘 수 있으므로 실제 업무에서는 FP/FN 비용을 확인한 뒤 선택 규칙을 다시 정해야 합니다.

## 8. Final Test와 FP/FN 해석

Validation에서 **Logistic Regression + Threshold 0.20**을 고정한 뒤 Final Test를 한 번 평가했습니다.

Final Test:
- Accuracy: **0.280**
- Precision: **0.244**
- Recall: **0.846**
- F1: **0.379**

Confusion Matrix:
- TN = **3**
- FP = **34**
- FN = **2**
- TP = **11**

Recall은 높았지만 완료 주문 37건 중 34건을 취소 위험으로 잘못 분류했습니다. 따라서 취소 주문을 많이 잡는 대신 정상 주문에 대한 오탐이 매우 크게 늘어난 결과입니다.

In [9]:
y_pred_test, y_proba_test, test_metrics = final_test_evaluation(
    selected_model,
    X_test,
    y_test,
    threshold=selected_threshold,
)

confusion_df = confusion_matrix_dataframe(y_test, y_pred_test)

display(test_metrics)
display(confusion_df)

tn = int(confusion_df.loc["actual_completed", "pred_completed"])
fp = int(confusion_df.loc["actual_completed", "pred_cancelled"])
fn = int(confusion_df.loc["actual_cancelled", "pred_completed"])
tp = int(confusion_df.loc["actual_cancelled", "pred_cancelled"])
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(confusion_df.values)
for i in range(confusion_df.shape[0]):
    for j in range(confusion_df.shape[1]):
        ax.text(j, i, int(confusion_df.iloc[i, j]), ha="center", va="center")
ax.set_xticks([0, 1], labels=["Pred Completed", "Pred Cancelled"])
ax.set_yticks([0, 1], labels=["Actual Completed", "Actual Cancelled"])
ax.set_title("Final Test Confusion Matrix")
fig.colorbar(im, ax=ax)
fig.tight_layout()
fig.savefig(IMAGE_DIR / "step06_confusion_matrix.png", dpi=150)
plt.show()

,evaluation_split,threshold,selection_status,accuracy,precision,recall,f1
0,test,0.2,frozen_before_test,0.28,0.244444,0.846154,0.37931


,pred_completed,pred_cancelled
actual_completed,3,34
actual_cancelled,2,11


TN=3, FP=34, FN=2, TP=11


### 업무·분석적 의미와 한계

현재 Threshold에서는 FN이 **2건**으로 적은 대신 FP가 **34건** 발생했습니다. 취소 위험 선별이 목적이라도 정상 완료 주문에 지나치게 많은 확인·개입이 발생할 수 있습니다.

어느 오류를 더 줄일지는 **개입 비용, 취소 손실액, 고객 경험, 담당자 처리 용량** 같은 업무 정보가 추가로 필요합니다. 현재 데이터에는 실제 비용 정보가 없으므로 FP와 FN 중 어느 쪽이 절대적으로 더 비싸다고 단정할 수 없습니다.

또한 모델이 학습한 것은 **예측 패턴**이지 취소의 원인이 아닙니다.

## 9. Internal / Public 결과 구분

오류 분석을 위한 내부 결과에는 source index 같은 추적 정보가 필요할 수 있지만, 공개 Prediction에서는 식별자를 제거합니다.

공개 컬럼:
`record_id, actual_is_cancelled, predicted_is_cancelled, cancel_probability, model, threshold`

In [10]:
prediction_internal = create_prediction_result(
    source_index=X_test.index,
    y_test=y_test,
    y_pred=y_pred_test,
    y_proba=y_proba_test,
    model_name=selected_model_name,
    threshold=selected_threshold,
)
prediction_public = public_prediction_result(prediction_internal)

display(prediction_public.head())
print("Public columns:", prediction_public.columns.tolist())

forbidden_public = {"source_index", "order_id", "customer_id", "product_id"}
assert not forbidden_public.intersection(prediction_public.columns)
print("Privacy-safe Output: PASS")

,record_id,actual_is_cancelled,predicted_is_cancelled,cancel_probability,model,threshold
0,test_0001,0,1,0.576120,Logistic Regression,0.2
1,test_0002,0,1,0.462559,Logistic Regression,0.2
2,test_0003,0,1,0.406685,Logistic Regression,0.2
3,test_0004,0,1,0.829364,Logistic Regression,0.2
4,test_0005,0,1,0.883050,Logistic Regression,0.2


Public columns: ['record_id', 'actual_is_cancelled', 'predicted_is_cancelled', 'cancel_probability', 'model', 'threshold']
Privacy-safe Output: PASS


## 10. Validation Evidence와 전체 재실행

자동 Validation 결과는 다음 핵심 계약에서 **모두 PASS**했습니다.

- Target이 completed/cancelled의 0·1 이진 범위인지
- 금지 Feature가 입력에 섞이지 않았는지
- 병합 행 수와 미매칭이 정상인지
- Train/Validation/Test에 두 클래스가 모두 존재하는지
- 모델이 Validation에서 선택됐는지
- Threshold가 Validation에서 선택됐는지
- Public prediction에 식별자가 노출되지 않았는지

자동 PASS는 **코드 계약이 지켜졌다는 뜻**입니다. 운영 적합성, 공정성, 시간에 따른 성능 변화까지 증명하는 것은 아닙니다. 특히 이번 Final Test 성능과 날짜 품질 문제는 별도로 판단해야 합니다.

In [11]:
validation = build_classification_validation(
    model_data=model_data,
    features=features,
    merge_checks=merge_checks,
    y_train=y_train,
    y_valid=y_valid,
    y_test=y_test,
    validation_comparison=validation_comparison,
    selected_model_name=selected_model_name,
    threshold_df=threshold_df,
    selected_threshold=selected_threshold,
    prediction_result_public=prediction_public,
)
display(validation)
assert validation["status"].eq("PASS").all()
print("Validation Evidence: ALL PASS")

,check,value,status
0,target_exactly_completed_cancelled,"[0, 1]",PASS
1,forbidden_feature_overlap,0,PASS
2,merge_rows_and_matches,True,PASS
3,all_splits_have_both_classes,True,PASS
4,model_selected_on_validation,True,PASS
5,threshold_selected_on_validation,True,PASS
6,public_prediction_identifier_columns,none,PASS


Validation Evidence: ALL PASS


In [12]:
result = run_classification_analysis(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    random_state=42,
)

print("재실행 선택 모델:", result["selected_model_name"])
print("재실행 Threshold:", result["selected_threshold"])
display(result["test_metrics"])
display(result["validation"])

assert result["selected_model_name"] == selected_model_name
assert np.isclose(result["selected_threshold"], selected_threshold)
assert result["validation"]["status"].eq("PASS").all()
print("전체 재실행: PASS")

재실행 선택 모델: Logistic Regression
재실행 Threshold: 0.2


,evaluation_split,threshold,selection_status,accuracy,precision,recall,f1
0,test,0.2,frozen_before_test,0.28,0.244444,0.846154,0.37931


,check,value,status
0,target_exactly_completed_cancelled,"[0, 1]",PASS
1,forbidden_feature_overlap,0,PASS
2,merge_rows_and_matches,True,PASS
3,all_splits_have_both_classes,True,PASS
4,model_selected_on_validation,True,PASS
5,threshold_selected_on_validation,True,PASS
6,public_prediction_identifier_columns,none,PASS


전체 재실행: PASS


## 11. 최종 사용 판단

**현재 판단: 현재 사용 보류**

### 판단 근거

1. Validation에서는 Logistic Regression이 선택됐고 Threshold 0.20에서 F1 **0.456**, Recall **1.000**이었지만, Final Test에서는 F1이 **0.379**로 낮아졌습니다.
2. Final Test Accuracy가 **0.280**, Precision이 **0.244**이며, 완료 주문 37건 중 **34건이 FP**였습니다. 현재 상태로 자동 개입에 사용하면 오탐 부담이 너무 큽니다.
3. 누수·병합·선택 절차·Privacy 관련 자동 Evidence는 모두 PASS했지만, 이 PASS는 운영 성능을 보장하지 않습니다.
4. 이진 대상 248건 중 **45건의 `signup_date > order_date` 시간 모순**이 있어 원천 데이터 품질 확인도 필요합니다.

### 현재 모델의 가장 큰 위험

낮은 Threshold가 취소 Recall을 높이는 대신 정상 주문을 대량으로 취소 위험으로 분류한다는 점입니다. 또한 현재 split은 교육용 random split이라 미래 주문에서도 같은 성능이 유지된다고 보장할 수 없습니다.

### 다음 개선 우선순위

1. 가입일·주문일 시간 모순의 원천 데이터 정정
2. 시간 순서 기반 out-of-time 평가
3. 실제 FP/FN 비용을 반영한 Threshold 정책
4. 기간·고객군별 성능과 공정성 점검

### 예측과 인과 구분

현재 확인한 것은 주어진 Feature 조합과 주문 취소 사이의 **예측 패턴**입니다. 특정 결제수단, 고객 특성, 주문 금액 등이 취소를 **원인적으로 발생시킨다**고는 말할 수 없습니다.

## 12. 최종 체크

- [x] `completed=0`, `cancelled=1` 타깃 범위를 정의했습니다.
- [x] `refunded`와 기타 상태를 제외했습니다.
- [x] 예측 시점과 Feature Contract를 정의했습니다.
- [x] target, ID, 사후 정보를 feature에서 제외했습니다.
- [x] `line_total = quantity × unit_price`를 검증했습니다.
- [x] 주문 단위 특징과 병합 관계를 검증했습니다.
- [x] 클래스 비율을 확인했습니다.
- [x] Dummy baseline과 비교했습니다.
- [x] Train / Validation / Test 역할을 구분했습니다.
- [x] 모델을 Validation에서 선택했습니다.
- [x] Threshold를 Validation에서 선택했습니다.
- [x] Final Test 전에 선택을 고정했습니다.
- [x] Precision / Recall / F1을 함께 확인했습니다.
- [x] FP/FN의 의미를 설명했습니다.
- [x] Public prediction에서 식별자를 제거했습니다.
- [x] Validation Evidence를 확인했습니다.
- [x] random split의 한계를 기록했습니다.
- [x] 예측 패턴을 원인처럼 단정하지 않았습니다.

최종 제출 파일: `assignments/chapter10/chapter10.ipynb`